# Notebook 1/2 — ประเมินผลโมเดล YOLO ด้วย Metric มาตรฐาน (Built-in Evaluation)

สมุดบันทึกนี้ใช้ฟังก์ชัน `model.val()` ของไลบรารี **Ultralytics** เพื่อประเมินผลโมเดลบนชุดข้อมูล **test**
โดยจะได้ค่า **Precision, Recall, mAP@50, mAP@50-95** ที่คำนวณตามมาตรฐานของ YOLO/COCO

> ไฟล์นี้เป็นไฟล์ที่ 1 จาก 2 ไฟล์ — ไฟล์ที่ 2 (`02_custom_iou_confusion_matrix.ipynb`) จะคำนวณ
> **IoU, Confusion Matrix, F1-score และ Accuracy = TP/(TP+FP+FN)** แบบกำหนดเอง (custom) โดยไม่พึ่งพา built-in ของ ultralytics
> เพื่อให้เห็นกระบวนการคำนวณอย่างชัดเจน สามารถนำสมการไปอ้างอิงในเล่มวิทยานิพนธ์ได้

---

## 1. สรุปสมการ (Equations)

**Intersection over Union (IoU)**

$$IoU(B_{pred}, B_{gt}) = \dfrac{Area(B_{pred} \cap B_{gt})}{Area(B_{pred} \cup B_{gt})}$$

ค่ากล่องที่ทำนาย (Prediction) จะถูกนับเป็น **True Positive (TP)** ก็ต่อเมื่อ IoU กับกล่อง Ground Truth
ในคลาสเดียวกัน **≥ threshold** ที่กำหนด (เช่น 0.5 สำหรับ mAP@50)

**Precision** — สัดส่วนของกล่องที่ทำนายถูกต้อง เทียบกับกล่องที่ทำนายทั้งหมด

$$Precision = \dfrac{TP}{TP + FP}$$

**Recall** — สัดส่วนของวัตถุจริงที่โมเดลตรวจจับเจอ เทียบกับวัตถุจริงทั้งหมด

$$Recall = \dfrac{TP}{TP + FN}$$

**Average Precision (AP)** — พื้นที่ใต้กราฟ Precision–Recall curve ของแต่ละคลาส

$$AP = \int_{0}^{1} P(r)\, dr$$

**mAP@50** — ค่าเฉลี่ย AP ของทุกคลาส โดยใช้ IoU threshold คงที่ที่ 0.5

$$mAP@50 = \dfrac{1}{N_{class}}\sum_{c=1}^{N_{class}} AP_c \Big|_{IoU=0.5}$$

**mAP@50-95** — ค่าเฉลี่ย AP ของทุกคลาส โดยเฉลี่ยซ้ำที่ IoU threshold ตั้งแต่ 0.50 ถึง 0.95 (step 0.05, รวม 10 ค่า)
ตามมาตรฐาน COCO

$$mAP@50\text{-}95 = \dfrac{1}{10}\sum_{t \in \{0.50,0.55,...,0.95\}} \left( \dfrac{1}{N_{class}}\sum_{c=1}^{N_{class}} AP_c \Big|_{IoU=t} \right)$$

---


## 2. ติดตั้งไลบรารีที่จำเป็น

## 3. ตั้งค่า Path (แก้ไขให้ตรงกับโครงสร้างไฟล์ของคุณ)

In [ ]:
from pathlib import Path

# --- แก้ไข path ตรงนี้ให้ตรงกับเครื่องของคุณ ---
MODEL_PATH = "/plateLDR_640/weights/best.pt"   # path ไปยังไฟล์น้ำหนักโมเดลที่เทรนแล้ว (.pt)
DATA_YAML  = "/evaluation_img/data.yaml"                            # path ไปยังไฟล์ yaml เช่น
#   nc: 2
#   names: ['car', 'license-plate']
#   train: ...
#   val:   ...
#   test:  test/images        <-- ต้องมี key นี้ชี้ไปยังโฟลเดอร์รูปภาพชุด test

SPLIT = "test"     # ใช้ split ที่ชื่อ 'test' ตามที่กำหนดใน data.yaml
IOU_THRES_NMS = 0.7
CONF_THRES = 0.001  # ใช้ค่าต่ำตอนประเมินผล (มาตรฐานของ YOLO val) เพื่อให้คำนวณ PR-curve ได้ครบช่วง

assert Path(MODEL_PATH).exists(), f"ไม่พบไฟล์โมเดล: {MODEL_PATH}"
assert Path(DATA_YAML).exists(), f"ไม่พบไฟล์ yaml: {DATA_YAML}"
print("Model:", MODEL_PATH)
print("Data yaml:", DATA_YAML)


## 4. โหลดโมเดล และรันการประเมินผล (Validation) บน test set

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

metrics = model.val(
    data=DATA_YAML,
    split=SPLIT,
    conf=CONF_THRES,
    iou=IOU_THRES_NMS,
    save_json=True,
    plots=True,          # จะ save รูป PR curve, F1 curve, confusion matrix ให้อัตโนมัติในโฟลเดอร์ runs/detect/val*
)


## 5. ดึงค่า Metric หลักออกมา

`metrics.box` เก็บค่าที่ ultralytics คำนวณให้แล้วตามสมการในหัวข้อ 1


In [ ]:
import numpy as np

precision_per_class = metrics.box.p          # precision ต่อคลาส (ที่ IoU/conf จุดที่ดีที่สุด)
recall_per_class    = metrics.box.r          # recall ต่อคลาส
ap50_per_class       = metrics.box.ap50       # AP ที่ IoU=0.5 ต่อคลาส
ap_per_class         = metrics.box.ap         # AP เฉลี่ยที่ IoU=0.5:0.95 ต่อคลาส
class_names          = metrics.names

print(f"{'Class':<18}{'Precision':>12}{'Recall':>12}{'AP@50':>12}{'AP@50-95':>12}")
for i, name in class_names.items():
    p  = precision_per_class[i] if i < len(precision_per_class) else float('nan')
    r  = recall_per_class[i]    if i < len(recall_per_class) else float('nan')
    a50 = ap50_per_class[i]     if i < len(ap50_per_class) else float('nan')
    a   = ap_per_class[i]       if i < len(ap_per_class) else float('nan')
    print(f"{name:<18}{p:>12.4f}{r:>12.4f}{a50:>12.4f}{a:>12.4f}")

print("\n===== ภาพรวม (mean over classes) =====")
print(f"Precision (mean)   = {metrics.box.mp:.4f}")
print(f"Recall    (mean)   = {metrics.box.mr:.4f}")
print(f"mAP@50             = {metrics.box.map50:.4f}")
print(f"mAP@50-95          = {metrics.box.map:.4f}")

# F1-score เสริม จาก Precision/Recall ที่ได้ (ตามสมการ F1 ในไฟล์ที่ 2)
mean_p, mean_r = metrics.box.mp, metrics.box.mr
f1_overall = 2 * mean_p * mean_r / (mean_p + mean_r + 1e-16)
print(f"F1-score  (mean)   = {f1_overall:.4f}")


## 6. บันทึกผลสรุปเป็นตาราง CSV (สำหรับแนบในเล่ม)

In [ ]:
import pandas as pd

rows = []
for i, name in class_names.items():
    rows.append({
        "class": name,
        "precision": precision_per_class[i] if i < len(precision_per_class) else np.nan,
        "recall": recall_per_class[i] if i < len(recall_per_class) else np.nan,
        "AP@50": ap50_per_class[i] if i < len(ap50_per_class) else np.nan,
        "AP@50-95": ap_per_class[i] if i < len(ap_per_class) else np.nan,
    })
rows.append({
    "class": "ALL (mean)",
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "AP@50": metrics.box.map50,
    "AP@50-95": metrics.box.map,
})

df = pd.DataFrame(rows)
df.to_csv("builtin_metrics_summary.csv", index=False)
df


## 7. รูปกราฟที่ ultralytics สร้างให้อัตโนมัติ

หลังรันเซลล์ข้อ 4 แล้ว ultralytics จะบันทึกไฟล์ต่อไปนี้ไว้ในโฟลเดอร์ `runs/detect/val*/` โดยอัตโนมัติ
สามารถนำไปแนบในเล่มได้เลย:
- `confusion_matrix.png`, `confusion_matrix_normalized.png`
- `P_curve.png` (Precision-Confidence curve)
- `R_curve.png` (Recall-Confidence curve)
- `PR_curve.png` (Precision-Recall curve — พื้นที่ใต้กราฟ = AP)
- `F1_curve.png`


In [ ]:
import glob
from IPython.display import Image, display

save_dir = str(metrics.save_dir)
print("ผลลัพธ์กราฟถูกบันทึกไว้ที่:", save_dir)

for img_path in sorted(glob.glob(f"{save_dir}/*curve*.png")) + sorted(glob.glob(f"{save_dir}/confusion_matrix*.png")):
    print(img_path)
    display(Image(filename=img_path, width=500))
